# MCP tools, agents, and AssetOpsBench architecture

> **KDD tutorial — complete introductory version.** This is the canonical notebook. It includes the industrial motivation, MCP fundamentals, MCP tool lifecycle, AssetOpsBench benchmark architecture, direct tool execution, agent trajectories, and execution safety.

An introduction for the KDD tutorial. No previous knowledge of industrial asset operations, MCP, or AI agents is assumed. The notebook first explains the problem and vocabulary, then uses the real AssetOpsBench Utilities MCP server. The core exercises require no model credentials and perform no database writes.

## Learning objectives

By the end of this notebook, you should be able to:

1. explain why industrial agents need grounded tools rather than model knowledge alone;
2. describe AssetOpsBench and the purpose of its scenarios, MCP servers, trajectories, evaluation, and leaderboard;
3. distinguish an LLM, an agent, an MCP host, an MCP client, and an MCP server;
4. explain tools, resources, and prompts;
5. read a tool definition: name, description, input schema, and result;
6. discover a server's live tool contract and call a tool directly;
7. explain the agent tool-calling loop;
8. distinguish tools-only, code-enabled, and read-only execution.




## Before you begin

New to the repository? Complete [`00_environment_setup.ipynb`](00_environment_setup.ipynb) first. It covers Python, `uv`, `.env`, Docker, CouchDB data, and selecting the correct notebook kernel.

The check below is intentionally read-only. The conceptual and Utilities sections can run without CouchDB or model credentials; later IoT, FMSR, Work Order, TSFM, Vibration, and Stirrup exercises have additional requirements described in the setup notebook.


In [ ]:
from pathlib import Path
import importlib.util
import shutil
import socket
import sys

def locate_assetopsbench(start=Path.cwd()):
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "src").is_dir():
            return candidate
    return None

def port_open(host, port, timeout=1.0):
    try:
        with socket.create_connection((host, port), timeout=timeout):
            return True
    except OSError:
        return False

INTRO_REPO = locate_assetopsbench()
INTRO_STATUS = {
    "repository": str(INTRO_REPO) if INTRO_REPO else "not found",
    "Python >= 3.12": sys.version_info >= (3, 12),
    "using .venv": Path(sys.prefix).name == ".venv",
    "uv available": shutil.which("uv") is not None,
    "MCP package available": importlib.util.find_spec("mcp") is not None,
    ".env present": bool(INTRO_REPO and (INTRO_REPO / ".env").exists()),
    "CouchDB reachable (optional here)": port_open("localhost", 5984),
}

for item, value in INTRO_STATUS.items():
    print(f"{item:38} {value}")

if not all([INTRO_STATUS["Python >= 3.12"], INTRO_STATUS["using .venv"], INTRO_STATUS["MCP package available"]]):
    print("\nComplete 00_environment_setup.ipynb before running the executable sections.")


## Why industrial asset operations need grounded AI

Industrial assets include chillers, pumps, motors, air-handling units, compressors, generators, and turbines. Maintenance and reliability teams combine several kinds of evidence:

- asset registries and installed sensors;
- live or historical time-series measurements;
- work orders and failure codes;
- failure-mode catalogs and maintenance knowledge;
- engineering rules and standards.

A language model may know general maintenance concepts, but it does **not** automatically know which assets are currently registered, what a sensor measured, which work order exists, or what records are stored in an organization's database. Asking it to answer from memory can produce plausible but ungrounded statements.

For operational use, we want the model to retrieve facts and run approved analyses through explicit capabilities. Those capabilities should have documented inputs, predictable outputs, permissions, and execution logs. This is the problem that MCP tools help solve.


## What is MCP?

The **Model Context Protocol (MCP)** is an open protocol for connecting AI applications to external capabilities and context through a consistent client-server interface. A useful analogy is a universal adapter: an AI host can connect to different servers without inventing a custom integration contract for every database, service, or analysis package.

MCP does **not** replace the model, the agent, the database, or the application logic. It standardizes communication between them. In this tutorial:

- the **host** runs the notebook or Stirrup agent and enforces policy;
- an **MCP client** maintains a session with one server;
- an **MCP server** advertises focused capabilities;
- the underlying system may be CouchDB, a file catalog, a clock, SciPy, or a time-series model.

MCP's data layer uses JSON-RPC 2.0 semantics. A local server can communicate over **stdio**; a remote server can use **Streamable HTTP**. A host normally creates one client connection per server.

Official references: [MCP architecture](https://modelcontextprotocol.io/docs/learn/architecture) and [MCP transports](https://modelcontextprotocol.io/specification/2025-11-25/basic/transports).


## MCP client-server architecture, visually

![Original AssetOpsBench MCP client-server architecture diagram](assets/mcp_client_server_architecture.png)

The diagram separates four responsibilities that are easy to confuse:

1. **MCP host / agent runner** — the user-facing application. In this tutorial, that can be notebook code or Stirrup. The model may propose an action, but the host controls execution, permissions, timeouts, and what results return to the model.
2. **MCP clients** — protocol connections managed by the host. A host normally maintains a separate client session for each server, so the IoT and Work Order connections are independent even when one agent uses both.
3. **MCP servers** — focused adapters that publish tool names, descriptions, JSON input schemas, and results. They contain or invoke the implementation that actually retrieves data or runs an analysis.
4. **Underlying systems** — databases, files, services, or analysis libraries accessed by a server. The LLM does not connect to CouchDB directly; it requests a tool, and the host executes that request through the appropriate client and server.

### Two deployment patterns

- **Local stdio:** the host launches a local MCP server subprocess and exchanges protocol messages over standard input/output. The KDD tutorial uses this pattern for the AssetOpsBench servers.
- **Remote Streamable HTTP:** the host connects to an MCP server exposed over HTTP. The server may then call an external service. This third lane is a generic deployment example, not the Utilities server used later in this notebook.

Both patterns use the same MCP data-layer concepts and JSON-RPC 2.0 semantics. The flow at the bottom is crucial: discover the live contract, select a tool, validate arguments and permission, execute it, and return the result as an observation. That observation may become a direct notebook result or additional context for the next turn of an agent.

> This is an original AssetOpsBench-specific redraw inspired by the architecture discussion in [Nir Diamant's MCP tutorial](https://github.com/NirDiamant/GenAI_Agents/blob/main/all_agents_tutorials/mcp-tutorial.ipynb). It uses current MCP transport terminology: **stdio** and **Streamable HTTP**.


## What exactly is an MCP tool?

An MCP tool is a server-exposed function that an AI application can discover and execute. Each tool contract normally includes:

| Part | Purpose | Example |
|---|---|---|
| **Name** | Stable identifier used in a call | `assets` |
| **Description** | Tells a model or developer when to use it | List assets at a site |
| **Input schema** | JSON Schema describing allowed and required arguments | `site_name: string` |
| **Result** | Grounded observation returned by execution | Asset IDs, types, and sensor counts |

The important separation is **proposal versus execution**. A model may propose `iot__assets(site_name="MAIN")`, but the host validates the request and the MCP server executes it. Text that merely looks like a tool call is not proof that anything ran.

### Tool lifecycle

```text
1. Host starts/connects to a server and initializes a session
2. Client requests tools/list
3. Server returns live names, descriptions, and schemas
4. A developer or model selects a tool and supplies arguments
5. Host applies permissions and validates the request
6. Client sends tools/call
7. Server runs the underlying implementation
8. Result returns as an observation
9. Direct code displays it, or an agent reasons over it and continues
```

This lifecycle gives us interoperability, grounding, permission boundaries, and an auditable record. It does not guarantee correctness by itself: tool descriptions, schemas, data quality, policies, and agent behavior still matter.


## What is AssetOpsBench?

[AssetOpsBench](https://github.com/IBM/AssetOpsBench) is an open framework and benchmark for **building, orchestrating, and evaluating AI agents for industrial asset operations and maintenance**. It is not one model and not one agent. It is a testbed containing the pieces needed to ask a reproducible question: *Can an agent complete an industrial task using the correct evidence and actions?*

| Benchmark layer | Role in AssetOpsBench |
|---|---|
| **Industrial environment** | Simulated asset, sensor, failure-mode, work-order, and time-series records |
| **MCP servers and tools** | Standardized access to Utilities, IoT, FMSR, Work Orders, TSFM, Vibration, and related capabilities |
| **Expert-curated scenarios** | Natural-language tasks authored from maintenance, reliability, facilities, and plant-operation perspectives |
| **Agent runners** | Different orchestration implementations, including Stirrup, execute the same tasks |
| **Trajectory** | Captures tool calls, exact inputs, outputs, timing, and the final response |
| **Evaluation** | Deterministic checks and rubric/LLM-based judging score both outcome and process |
| **Leaderboard** | Compares configurations reproducibly across a common scenario set |

Single-domain scenarios may list assets, retrieve failure modes, inspect work orders, choose time-series recipes, or assess vibration. End-to-end scenarios require an agent to combine multiple grounded results. The KDD tutorial walks through this stack from the bottom up: first direct MCP calls, then agent execution, then evaluation and leaderboard construction.


## AssetOpsBench benchmark pipeline

![AssetOpsBench benchmark architecture: MCP servers, scenarios, agent runners, trajectories, evaluation, and leaderboard](assets/assetopsbench_benchmark_architecture.png)

Read the figure from left to right:

1. **Domain systems become MCP capabilities.** Servers expose industrial records and analyses as discoverable tools.
2. **Experts define grounded scenarios.** Each query represents a realistic role and expected operational outcome.
3. **An agent performs the task.** The same scenario can be run with different models and orchestration frameworks.
4. **Execution produces a trajectory.** The trace distinguishes actual tool execution from unsupported claims.
5. **Evaluation scores answer and process.** Exact/numeric rules handle deterministic requirements; rubric-based judging handles semantic requirements.
6. **Leaderboards aggregate performance.** Comparisons are meaningful only when scenario set, capabilities, model, and execution mode are recorded.

> **Version note:** the supplied figure captures one project snapshot. Counts of scenarios, servers, tools, runners, and models evolve as the repository develops. Use the live repository and each server's `list_tools()` response as the source of truth.


## How the tutorial maps to the architecture

```text
00  Concepts: industrial problem → MCP → agents → AssetOpsBench
01  Utilities MCP: deterministic, credential-free tools
02  IoT MCP: sites, assets, and sensors
03  FMSR MCP: stored and generated failure modes
04  Work Order MCP: retrieval, catalogs, and work-order workflows
05  TSFM MCP: time-series evidence, model/feature catalogs, and recipes
06  Vibration MCP: deterministic engineering analysis
07  Stirrup end-to-end: a model chooses tools across servers
08  Evaluation: verify the answer and the trajectory
09  Leaderboard: aggregate comparable runs
```

The server notebooks deliberately call tools directly so that participants can see the contract and results without model variability. Notebook 07 is where Stirrup introduces model-controlled planning and tool selection.


## 1. The five pieces

| Piece | Responsibility | AssetOpsBench example |
|---|---|---|
| **LLM** | Produces language or structured tool calls from context | Claude, Llama, GPT |
| **Agent** | Repeatedly asks the model what to do, executes allowed actions, and returns observations | Stirrup agent |
| **MCP host** | Coordinates the model/agent and manages MCP clients, permissions, and context | Stirrup runner or this notebook |
| **MCP client** | Maintains one protocol session with one MCP server | `ClientSession` |
| **MCP server** | Exposes focused capabilities through a standard contract | Utilities, IoT, FMSR, Work Orders, TSFM, Vibration |

An LLM alone cannot query CouchDB or call a Python function. An agent can do so only when its host grants access to an appropriate tool. MCP standardizes the boundary between the host/client and those external capabilities.


## 2. MCP architecture

```text
User question
     │
     ▼
MCP host / agent runner
     │
     ├── LLM: decide, select tool, form arguments
     │
     ├── MCP client ──stdio──► Utilities MCP server ──► clock / local files
     ├── MCP client ──stdio──► IoT MCP server ───────► CouchDB
     ├── MCP client ──stdio──► FMSR MCP server ──────► CouchDB + optional LLM
     ├── MCP client ──stdio──► Work Order MCP server ► CouchDB
     ├── MCP client ──stdio──► TSFM MCP server ──────► files, catalogs, models
     └── MCP client ──stdio──► Vibration MCP server ─► deterministic diagnostics
```

A host creates a separate MCP client connection for each server. MCP messages use JSON-RPC semantics. Standard transports include local **stdio** and remote **Streamable HTTP**; AssetOpsBench uses local stdio for these tutorial servers.

Official references: [MCP architecture](https://modelcontextprotocol.io/docs/learn/architecture) and [MCP transports](https://modelcontextprotocol.io/specification/2025-11-25/basic/transports).


## 3. MCP primitives

Servers can expose three main primitives:

| Primitive | Meaning | Typical control | Example |
|---|---|---|---|
| **Tool** | Executable function with a named JSON input schema | Model-controlled, subject to host approval | Query assets or retrieve a work order |
| **Resource** | Addressable contextual data | Application-controlled | File contents or a database schema |
| **Prompt** | Reusable interaction template | User-controlled | A guided maintenance-analysis template |

AssetOpsBench's tutorial servers primarily expose **tools**. A tool definition contains a name, description, and input schema. A tool result is an observation—not automatically the final answer.


## 4. Environment setup

The repository already declares its dependencies. Do not install packages or paste API keys into notebook cells. Start VS Code/Jupyter from the AssetOpsBench repository and select its Python kernel.


In [1]:
from pathlib import Path
import json, os, shutil, subprocess, sys

def find_repo(start=Path.cwd()):
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "servers").exists():
            return candidate
    raise RuntimeError(
        "Open this notebook from inside the AssetOpsBench repository or change the "
        "VS Code/Jupyter working directory to the repository root."
    )

REPO = find_repo()
ARTIFACTS = REPO / "artifacts" / "kdd_tutorial"
ARTIFACTS.mkdir(parents=True, exist_ok=True)
assert shutil.which("uv"), "Install uv before continuing."
print("repository:", REPO)
print("python:", sys.version.split()[0])
print("uv:", shutil.which("uv"))


repository: /Users/chathurangishyalika/IBM/AssetOpsBench
python: 3.12.13
uv: /opt/homebrew/bin/uv


## 5. Connect to one real MCP server

The MCP client launches the Utilities server as a subprocess and communicates through standard input/output. `initialize()` negotiates the session before discovery or execution.


In [2]:
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

async def utilities_request(operation, tool_name=None, arguments=None):
    params = StdioServerParameters(
        command="uv",
        args=["run", "--directory", str(REPO), "utilities-mcp-server"],
        cwd=str(REPO),
        env=os.environ.copy(),
    )
    async with stdio_client(params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            if operation == "list":
                return await session.list_tools()
            return await session.call_tool(tool_name, arguments or {})

def parse_tool_result(result):
    text = "\n".join(getattr(item, "text", str(item)) for item in result.content)
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        return text


## 6. Discovery: ask the server what it can do

`list_tools()` retrieves the live contract. The server—not a slide, prompt, or old notebook—is the source of truth for tool names and arguments.


In [3]:
response = await utilities_request("list")
utilities_tools = [
    {
        "name": tool.name,
        "description": tool.description,
        "input_schema": tool.inputSchema,
    }
    for tool in response.tools
]
print("tools discovered:", len(utilities_tools))
utilities_tools


tools discovered: 6


[{'name': 'json_reader',
  'description': 'Reads a JSON file, parses its content, and returns the parsed data.',
  'input_schema': {'properties': {'file_name': {'title': 'File Name',
     'type': 'string'}},
   'required': ['file_name'],
   'title': 'json_readerArguments',
   'type': 'object'}},
 {'name': 'get_sensor_catalog',
  'description': 'Return cataloged sensor types.\n\n    Entries contain `sensor` and `description`. Omit sensor to list all cataloged\n    sensor types, or pass sensor for an exact sensor-name lookup.\n    ',
  'input_schema': {'properties': {'sensor': {'anyOf': [{'type': 'string'},
      {'type': 'null'}],
     'default': None,
     'title': 'Sensor'}},
   'title': 'get_sensor_catalogArguments',
   'type': 'object'}},
 {'name': 'get_asset_catalog',
  'description': 'Return cataloged asset classes and categories.\n\n    Entries contain `category`, `category_description`, `asset`, and\n    `description`. Omit filters to list all cataloged asset classes, or pass\n 

### Read a tool schema

The schema answers three questions: What is the tool called? What does it do? Which arguments are accepted or required? Agents receive this same information when deciding how to act.


In [4]:
contract = {tool["name"]: tool["input_schema"] for tool in utilities_tools}
for name, schema in contract.items():
    properties = list(schema.get("properties", {}))
    required = schema.get("required", [])
    print(f"{name}: arguments={properties}, required={required}")


json_reader: arguments=['file_name'], required=['file_name']
get_sensor_catalog: arguments=['sensor'], required=[]
get_asset_catalog: arguments=['asset', 'category'], required=[]
get_failure_mode_catalog: arguments=['failure_mode', 'category'], required=[]
current_date_time: arguments=[], required=[]
current_time_english: arguments=[], required=[]


## 7. Execution: call deterministic tools directly

Direct execution is useful for testing. No LLM or agent chooses the tool—the notebook author does.


In [5]:
now_result = await utilities_request("call", "current_date_time", {})
now = parse_tool_result(now_result)
now


{'currentDateTime': '2026-08-06T16:29:06.362932Z',
 'currentDateTimeDescription': "Today's date is 2026-08-06 and time is 16:29:06."}

In [6]:
english_result = await utilities_request("call", "current_time_english", {})
english_time = parse_tool_result(english_result)
english_time


{'english': '2026-08-06 16:29:07', 'iso': '2026-08-06T16:29:07.254229Z'}

### A tool with an argument

This example creates a harmless tutorial artifact, then asks the MCP server to read it. The path is supplied according to the discovered schema.


In [7]:
sample_file = ARTIFACTS / "mcp_intro_sample.json"
sample_file.write_text(
    json.dumps({"tutorial": "KDD", "topic": "MCP", "ready": True}, indent=2),
    encoding="utf-8",
)
read_result = await utilities_request(
    "call", "json_reader", {"file_name": str(sample_file)}
)
parsed_sample = parse_tool_result(read_result)
parsed_sample


{'tutorial': 'KDD', 'topic': 'MCP', 'ready': True}

## 8. What happened on the wire?

The SDK handled JSON-RPC messages for us. Conceptually, discovery and execution look like this:

```json
{"jsonrpc": "2.0", "id": 1, "method": "tools/list"}
```

```json
{
  "jsonrpc": "2.0",
  "id": 2,
  "method": "tools/call",
  "params": {
    "name": "json_reader",
    "arguments": {"file_name": ".../mcp_intro_sample.json"}
  }
}
```

You normally use the SDK rather than manually constructing these messages.


## 9. Direct MCP client versus agent

The cells above are **not an agent**. They follow a predetermined program:

```text
Notebook code → call current_date_time → display result
```

An agent introduces a model-controlled decision loop:

```text
1. Receive user question
2. Inspect available tool schemas
3. Ask the LLM for the next action
4. If it requests a tool, validate permissions and arguments
5. Execute the tool through MCP
6. Append the observation to context
7. Repeat until the model returns a final answer or a limit is reached
```

The host—not the model—actually executes tools. This separation lets the host enforce permissions, timeouts, logging, and user consent.


## 10. Tool-grounded trajectories

AssetOpsBench persists a trajectory so evaluation can distinguish real execution from a model merely printing a proposed call. A simplified valid turn looks like:

```json
{
  "index": 0,
  "text": "I will retrieve the record.",
  "tool_calls": [
    {
      "name": "wo__get_workorder",
      "input": {"site_id": "NORTH", "wonum": "1000050"},
      "output": "{...actual MCP result...}"
    }
  ]
}
```

A tool name written only inside `text` is not execution. Evaluation should inspect `tool_calls`, exact inputs, outputs, and forbidden actions.


## 11. Three independent safety dimensions

| Setting | Meaning | Does it guarantee read-only behavior? |
|---|---|---:|
| **Tools-only** (`--no-code`) | Agent has MCP tools but no generated-code execution tool | No |
| **Code-enabled** | Agent may generate and execute code in an isolated environment | No |
| **Read-only server** (`AOB_READONLY=1`) | Mutation tools are not registered by a server that supports this flag | Yes, for that server's exposed contract |

A tools-only agent can still call a write MCP tool if the server exposes one. For the final work-order example, we use both `--no-code` and `AOB_READONLY=1`. Code-enabled and tools-only runs belong on separate leaderboard tracks because their capabilities differ.


## 12. Optional: let Stirrup choose and call a tool

This is the only model-dependent section. It is disabled by default so the first-hour core remains reliable. Stirrup runs in tools-only mode. Configure a known working tool-calling model and set `KDD_RUN_INTRO_AGENT=1` before starting Jupyter.


In [8]:
RUN_INTRO_AGENT = os.getenv("KDD_RUN_INTRO_AGENT", "0") == "1"
MODEL_ID = os.getenv("KDD_MODEL_ID", "litellm_proxy/aws/claude-opus-4-8")
AGENT_TIMEOUT_SECONDS = int(os.getenv("KDD_AGENT_TIMEOUT_SECONDS", "180"))

if RUN_INTRO_AGENT:
    if MODEL_ID.startswith("litellm_proxy/"):
        missing = [name for name in ("LITELLM_API_KEY", "LITELLM_BASE_URL") if not os.getenv(name)]
    elif MODEL_ID.startswith("watsonx/"):
        missing = [name for name in ("WATSONX_APIKEY", "WATSONX_PROJECT_ID") if not os.getenv(name)]
    else:
        missing = []
    if missing:
        raise RuntimeError("Missing model configuration: " + ", ".join(missing))

    prompt = (
        "Use utilities__current_date_time to retrieve the current date and time. "
        "You must execute the tool once. Return only the tool-grounded result."
    )
    cmd = [
        "uv", "run", "--directory", str(REPO), "stirrup-agent",
        "--no-code", "--show-trajectory", "--max-turns", "3",
        "--model-id", MODEL_ID, prompt,
    ]
    completed = subprocess.run(
        cmd, env=os.environ.copy(), text=True, capture_output=True,
        timeout=AGENT_TIMEOUT_SECONDS,
    )
    print(completed.stdout)
    if completed.returncode:
        print(completed.stderr)
    completed.check_returncode()
else:
    print("Optional Stirrup call skipped. Direct MCP exercises are complete.")


Optional Stirrup call skipped. Direct MCP exercises are complete.


## 13. Knowledge check

1. Who chooses a tool in a direct client example? **The notebook author.**
2. Who proposes a tool in an agent example? **The model, within the agent loop.**
3. Who actually executes the tool? **The host through its MCP client.**
4. Where does the tool implementation live? **In the MCP server.**
5. Does `--no-code` prevent database writes? **No; use server permissions/read-only configuration.**
6. Why call `list_tools()`? **To discover the live contract and avoid stale tool names or arguments.**

## Next

Continue to notebook 01 for the full Utilities walkthrough, then progress through IoT, FMSR, Work Orders, TSFM, Vibration, Stirrup, evaluation, and leaderboard construction.
